In [1]:
library(forecast)
library(M4comp2018)
library(dplyr)

Registered S3 method overwritten by 'quantmod':
  method            from
  as.zoo.data.frame zoo 


载入程辑包：‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




In [2]:
nnetarl=load('Quarterly_nnetar_datalist.RData')
nnetar_datalist <- eval(parse(text = nnetarl))

In [3]:
nnetar_predh=nnetar_datalist[[2]]

In [4]:
nnetar_pred_res=nnetar_datalist[[1]]

In [5]:
nnetaroptl=load('Quarterly_nnetar_opt_pre_result.RData')
nnetar_opt_pre_list<- eval(parse(text = nnetaroptl))

In [6]:
time_matrix <- matrix(0,ncol = 3, nrow =3)
colnames(time_matrix) <- c("user_time", "system_time", "elapsed_time")
start_time = Sys.time()

In [7]:
nnetar_pr_min=nnetar_opt_pre_list[[1]]
nnetar_pr_mean=nnetar_opt_pre_list[[2]]

In [8]:
head(nnetar_pr_min)

xgbclsmin,xgbregmin,lgbclsmin,lgbregmin
2,1.678847,4,1.507120
2,2.196492,3,1.596278
2,2.039880,0,1.354261
2,1.631919,2,1.392353
0,1.130262,4,1.363482
4,2.234442,4,1.973866


In [9]:
nnetar_pr_min=nnetar_pr_min+1
nnetar_pr_mean=nnetar_pr_mean+1



In [10]:
head(nnetar_pr_min)

xgbclsmin,xgbregmin,lgbclsmin,lgbregmin
3,2.678847,5,2.507120
3,3.196492,4,2.596278
3,3.039880,1,2.354261
3,2.631919,3,2.392353
1,2.130262,5,2.363482
5,3.234442,5,2.973866


In [11]:
round5=function(x)
    {
    l=round(x)
    for(i in 1:length(l))
    {
    if(l[i]>5)
        {
        l[i]=5
    }
    if(l[i]<1)
    {
        l[i]=1
    }
}
    return(l)
}

In [12]:
nnetar_pr_min[,2]=round5(nnetar_pr_min[,2])
nnetar_pr_min[,4]=round5(nnetar_pr_min[,4])

In [13]:
nnetar_pr_mean[,2]=round5(nnetar_pr_mean[,2])
nnetar_pr_mean[,4]=round5(nnetar_pr_mean[,4])

In [14]:
head(nnetar_pr_min)

xgbclsmin,xgbregmin,lgbclsmin,lgbregmin
3,3,5,3
3,3,4,3
3,3,1,2
3,3,3,2
1,2,5,2
5,3,5,3


In [15]:
Quarterly_M4 <- Filter(function(l) l$period == "Quarterly", M4)

In [16]:
data=Quarterly_M4

In [17]:
datalength=length(data)
h=data[[1]]$h
m=5
n=4
freq=frequency(data[[1]]$x)

In [18]:
realxx=matrix(0,datalength,h)
for(i in 1:datalength){
    realxx[i,]=data[[i]]$xx
}

In [19]:
length5=matrix(0,datalength,m)
for(k in 1:datalength)
{
y = data[[k]]$x
y_l=length(y)
loc = 1:length(y)
loc_m = as.integer(loc*m/length(y))
filt_d = data.frame(y,loc_m)
filt_0=filter(filt_d ,loc_m==0)
m_l=count(filt_0)
m_l=as.integer(m_l)
length5[k,]=c(0,1*m_l,2*m_l,3*m_l,4*m_l)
    }

In [20]:
end_time = Sys.time()
time_matrix[1,]=end_time-start_time

In [21]:
end_time-start_time

Time difference of 1.405771 mins

## nnetar

In [22]:
start_time = Sys.time()

In [23]:
set.seed(100)
index = sample(2,length(data),replace = TRUE,prob=c(0.7,0.3))
testindex=c(1:length(data))[index==2]

In [24]:
meanpre=function(pred_array,opt_pr,h,testindex)
    {
    data_length=length(testindex)
    meanpreh=matrix(0,data_length,h)
    for (i in 1:data_length){
    index=testindex[i]
    loc=opt_pr[index]
    predh=pred_array[index,loc,,]
    predh=unique(predh)
    meanpreh[i,]=apply(predh,2,mean)
}
    return(meanpreh)
}

In [25]:
predres=function(pred_matrix,data,opt_pr,lengthmatrix,h,testindex)
    {
    data_length=length(testindex)
    pred_res=matrix(0,data_length,6)
    model=ets(data[[1]]$x)
    fore_l=forecast(model,h=h)
    freq=frequency(data[[1]]$x)
    for (i in 1:data_length){
    index=testindex[i]
    loc=opt_pr[index]
    start=lengthmatrix[i,loc]
    y_all=data[[index]]$x
    y=y_all[start:length(y_all)]
    fore_l$x=ts(y,frequency=freq,end=end(y_all))
    real=data[[index]]$xx
    fore_l$mean=ts(pred_matrix[i,],start=start(real),frequency=freq)
    res=accuracy(fore_l,real)
    pred_res[i,]=res[2,1:6]
    }
    return(pred_res)
    }

In [26]:
predres_mul=function(pred_matrix,data,opt_pr_mul,lengthmatrix,h,testindex)
    {
    data_length=length(testindex)
    pred_res=matrix(0,data_length,6)
    model=ets(data[[1]]$x)
    fore_l=forecast(model,h=h)
    freq=frequency(data[[1]]$x)
    for (i in 1:data_length){
    index=testindex[i]
    loc=min(opt_pr_mul[index,])
    start=lengthmatrix[i,loc]
    y_all=data[[index]]$x
    y=y_all[start:length(y_all)]
    fore_l$x=ts(y,frequency=freq,end=end(y_all))
    real=data[[index]]$xx
    fore_l$mean=ts(pred_matrix[i,],start=start(real),frequency=freq)
    res=accuracy(fore_l,real)
    pred_res[i,]=res[2,1:6]
    }
    return(pred_res)
    }

In [27]:
nnetarxgbcls_min=meanpre(nnetar_predh,nnetar_pr_min[,1],h,testindex)
nnetarxgbreg_min=meanpre(nnetar_predh,nnetar_pr_min[,2],h,testindex)
nnetarlgbcls_min=meanpre(nnetar_predh,nnetar_pr_min[,3],h,testindex)
nnetarlgbreg_min=meanpre(nnetar_predh,nnetar_pr_min[,4],h,testindex)

In [28]:
nnetarxgbcls_mean=meanpre(nnetar_predh,nnetar_pr_mean[,1],h,testindex)
nnetarxgbreg_mean=meanpre(nnetar_predh,nnetar_pr_mean[,2],h,testindex)
nnetarlgbcls_mean=meanpre(nnetar_predh,nnetar_pr_mean[,3],h,testindex)
nnetarlgbreg_mean=meanpre(nnetar_predh,nnetar_pr_mean[,4],h,testindex)

### The error of each improved prediction result

In [29]:
lengthtest=length5[testindex,]

In [ ]:
nnetarminxgbclsres=predres(nnetarxgbcls_min,data,nnetar_pr_min[,1],lengthtest,h,testindex)
nnetarminxgbregres=predres(nnetarxgbreg_min,data,nnetar_pr_min[,2],lengthtest,h,testindex)
nnetarminlgbclsres=predres(nnetarlgbcls_min,data,nnetar_pr_min[,3],lengthtest,h,testindex)
nnetarminlgbregres=predres(nnetarlgbreg_min,data,nnetar_pr_min[,4],lengthtest,h,testindex)

Warning message in .cbind.ts(list(e1, e2), c(deparse(substitute(e1))[1L], deparse(substitute(e2))[1L]), :
“non-intersecting series”
Warning message in trainingaccuracy(object, test, d, D):
“test elements must be within sample”
Warning message in .cbind.ts(list(e1, e2), c(deparse(substitute(e1))[1L], deparse(substitute(e2))[1L]), :
“non-intersecting series”
Warning message in trainingaccuracy(object, test, d, D):
“test elements must be within sample”
Warning message in .cbind.ts(list(e1, e2), c(deparse(substitute(e1))[1L], deparse(substitute(e2))[1L]), :
“non-intersecting series”
Warning message in trainingaccuracy(object, test, d, D):
“test elements must be within sample”
Warning message in .cbind.ts(list(e1, e2), c(deparse(substitute(e1))[1L], deparse(substitute(e2))[1L]), :
“non-intersecting series”
Warning message in trainingaccuracy(object, test, d, D):
“test elements must be within sample”
Warning message in .cbind.ts(list(e1, e2), c(deparse(substitute(e1))[1L], deparse(substitute

Warning message in trainingaccuracy(object, test, d, D):
“test elements must be within sample”
Warning message in .cbind.ts(list(e1, e2), c(deparse(substitute(e1))[1L], deparse(substitute(e2))[1L]), :
“non-intersecting series”
Warning message in trainingaccuracy(object, test, d, D):
“test elements must be within sample”
Warning message in .cbind.ts(list(e1, e2), c(deparse(substitute(e1))[1L], deparse(substitute(e2))[1L]), :
“non-intersecting series”
Warning message in trainingaccuracy(object, test, d, D):
“test elements must be within sample”
Warning message in .cbind.ts(list(e1, e2), c(deparse(substitute(e1))[1L], deparse(substitute(e2))[1L]), :
“non-intersecting series”
Warning message in trainingaccuracy(object, test, d, D):
“test elements must be within sample”
Warning message in .cbind.ts(list(e1, e2), c(deparse(substitute(e1))[1L], deparse(substitute(e2))[1L]), :
“non-intersecting series”
Warning message in trainingaccuracy(object, test, d, D):
“test elements must be within sampl

Warning message in .cbind.ts(list(e1, e2), c(deparse(substitute(e1))[1L], deparse(substitute(e2))[1L]), :
“non-intersecting series”
Warning message in trainingaccuracy(object, test, d, D):
“test elements must be within sample”
Warning message in .cbind.ts(list(e1, e2), c(deparse(substitute(e1))[1L], deparse(substitute(e2))[1L]), :
“non-intersecting series”
Warning message in trainingaccuracy(object, test, d, D):
“test elements must be within sample”
Warning message in .cbind.ts(list(e1, e2), c(deparse(substitute(e1))[1L], deparse(substitute(e2))[1L]), :
“non-intersecting series”
Warning message in trainingaccuracy(object, test, d, D):
“test elements must be within sample”
Warning message in .cbind.ts(list(e1, e2), c(deparse(substitute(e1))[1L], deparse(substitute(e2))[1L]), :
“non-intersecting series”
Warning message in trainingaccuracy(object, test, d, D):
“test elements must be within sample”
Warning message in .cbind.ts(list(e1, e2), c(deparse(substitute(e1))[1L], deparse(substitute

Warning message in trainingaccuracy(object, test, d, D):
“test elements must be within sample”
Warning message in .cbind.ts(list(e1, e2), c(deparse(substitute(e1))[1L], deparse(substitute(e2))[1L]), :
“non-intersecting series”
Warning message in trainingaccuracy(object, test, d, D):
“test elements must be within sample”
Warning message in .cbind.ts(list(e1, e2), c(deparse(substitute(e1))[1L], deparse(substitute(e2))[1L]), :
“non-intersecting series”
Warning message in trainingaccuracy(object, test, d, D):
“test elements must be within sample”
Warning message in .cbind.ts(list(e1, e2), c(deparse(substitute(e1))[1L], deparse(substitute(e2))[1L]), :
“non-intersecting series”
Warning message in trainingaccuracy(object, test, d, D):
“test elements must be within sample”
Warning message in .cbind.ts(list(e1, e2), c(deparse(substitute(e1))[1L], deparse(substitute(e2))[1L]), :
“non-intersecting series”
Warning message in trainingaccuracy(object, test, d, D):
“test elements must be within sampl

Warning message in .cbind.ts(list(e1, e2), c(deparse(substitute(e1))[1L], deparse(substitute(e2))[1L]), :
“non-intersecting series”
Warning message in trainingaccuracy(object, test, d, D):
“test elements must be within sample”
Warning message in .cbind.ts(list(e1, e2), c(deparse(substitute(e1))[1L], deparse(substitute(e2))[1L]), :
“non-intersecting series”
Warning message in trainingaccuracy(object, test, d, D):
“test elements must be within sample”
Warning message in .cbind.ts(list(e1, e2), c(deparse(substitute(e1))[1L], deparse(substitute(e2))[1L]), :
“non-intersecting series”
Warning message in trainingaccuracy(object, test, d, D):
“test elements must be within sample”
Warning message in .cbind.ts(list(e1, e2), c(deparse(substitute(e1))[1L], deparse(substitute(e2))[1L]), :
“non-intersecting series”
Warning message in trainingaccuracy(object, test, d, D):
“test elements must be within sample”
Warning message in .cbind.ts(list(e1, e2), c(deparse(substitute(e1))[1L], deparse(substitute

...

Warning message in trainingaccuracy(object, test, d, D):
“test elements must be within sample”
Warning message in .cbind.ts(list(e1, e2), c(deparse(substitute(e1))[1L], deparse(substitute(e2))[1L]), :
“non-intersecting series”
Warning message in trainingaccuracy(object, test, d, D):
“test elements must be within sample”
Warning message in .cbind.ts(list(e1, e2), c(deparse(substitute(e1))[1L], deparse(substitute(e2))[1L]), :
“non-intersecting series”
Warning message in trainingaccuracy(object, test, d, D):
“test elements must be within sample”
Warning message in .cbind.ts(list(e1, e2), c(deparse(substitute(e1))[1L], deparse(substitute(e2))[1L]), :
“non-intersecting series”
Warning message in trainingaccuracy(object, test, d, D):
“test elements must be within sample”
Warning message in .cbind.ts(list(e1, e2), c(deparse(substitute(e1))[1L], deparse(substitute(e2))[1L]), :
“non-intersecting series”
Warning message in trainingaccuracy(object, test, d, D):
“test elements must be within sampl

Warning message in .cbind.ts(list(e1, e2), c(deparse(substitute(e1))[1L], deparse(substitute(e2))[1L]), :
“non-intersecting series”
Warning message in trainingaccuracy(object, test, d, D):
“test elements must be within sample”
Warning message in .cbind.ts(list(e1, e2), c(deparse(substitute(e1))[1L], deparse(substitute(e2))[1L]), :
“non-intersecting series”
Warning message in trainingaccuracy(object, test, d, D):
“test elements must be within sample”
Warning message in .cbind.ts(list(e1, e2), c(deparse(substitute(e1))[1L], deparse(substitute(e2))[1L]), :
“non-intersecting series”
Warning message in trainingaccuracy(object, test, d, D):
“test elements must be within sample”
Warning message in .cbind.ts(list(e1, e2), c(deparse(substitute(e1))[1L], deparse(substitute(e2))[1L]), :
“non-intersecting series”
Warning message in trainingaccuracy(object, test, d, D):
“test elements must be within sample”
Warning message in .cbind.ts(list(e1, e2), c(deparse(substitute(e1))[1L], deparse(substitute

Warning message in trainingaccuracy(object, test, d, D):
“test elements must be within sample”
Warning message in .cbind.ts(list(e1, e2), c(deparse(substitute(e1))[1L], deparse(substitute(e2))[1L]), :
“non-intersecting series”
Warning message in trainingaccuracy(object, test, d, D):
“test elements must be within sample”
Warning message in .cbind.ts(list(e1, e2), c(deparse(substitute(e1))[1L], deparse(substitute(e2))[1L]), :
“non-intersecting series”
Warning message in trainingaccuracy(object, test, d, D):
“test elements must be within sample”
Warning message in .cbind.ts(list(e1, e2), c(deparse(substitute(e1))[1L], deparse(substitute(e2))[1L]), :
“non-intersecting series”
Warning message in trainingaccuracy(object, test, d, D):
“test elements must be within sample”
Warning message in .cbind.ts(list(e1, e2), c(deparse(substitute(e1))[1L], deparse(substitute(e2))[1L]), :
“non-intersecting series”
Warning message in trainingaccuracy(object, test, d, D):
“test elements must be within sampl

Warning message in .cbind.ts(list(e1, e2), c(deparse(substitute(e1))[1L], deparse(substitute(e2))[1L]), :
“non-intersecting series”
Warning message in trainingaccuracy(object, test, d, D):
“test elements must be within sample”
Warning message in .cbind.ts(list(e1, e2), c(deparse(substitute(e1))[1L], deparse(substitute(e2))[1L]), :
“non-intersecting series”
Warning message in trainingaccuracy(object, test, d, D):
“test elements must be within sample”
Warning message in .cbind.ts(list(e1, e2), c(deparse(substitute(e1))[1L], deparse(substitute(e2))[1L]), :
“non-intersecting series”
Warning message in trainingaccuracy(object, test, d, D):
“test elements must be within sample”
Warning message in .cbind.ts(list(e1, e2), c(deparse(substitute(e1))[1L], deparse(substitute(e2))[1L]), :
“non-intersecting series”
Warning message in trainingaccuracy(object, test, d, D):
“test elements must be within sample”
Warning message in .cbind.ts(list(e1, e2), c(deparse(substitute(e1))[1L], deparse(substitute

Warning message in trainingaccuracy(object, test, d, D):
“test elements must be within sample”
Warning message in .cbind.ts(list(e1, e2), c(deparse(substitute(e1))[1L], deparse(substitute(e2))[1L]), :
“non-intersecting series”
Warning message in trainingaccuracy(object, test, d, D):
“test elements must be within sample”
Warning message in .cbind.ts(list(e1, e2), c(deparse(substitute(e1))[1L], deparse(substitute(e2))[1L]), :
“non-intersecting series”
Warning message in trainingaccuracy(object, test, d, D):
“test elements must be within sample”
Warning message in .cbind.ts(list(e1, e2), c(deparse(substitute(e1))[1L], deparse(substitute(e2))[1L]), :
“non-intersecting series”
Warning message in trainingaccuracy(object, test, d, D):
“test elements must be within sample”
Warning message in .cbind.ts(list(e1, e2), c(deparse(substitute(e1))[1L], deparse(substitute(e2))[1L]), :
“non-intersecting series”
Warning message in trainingaccuracy(object, test, d, D):
“test elements must be within sampl

In [ ]:
nnetarmeanxgbclsres=predres(nnetarxgbcls_mean,data,nnetar_pr_mean[,1],lengthtest,h,testindex)
nnetarmeanxgbregres=predres(nnetarxgbreg_mean,data,nnetar_pr_mean[,2],lengthtest,h,testindex)
nnetarmeanlgbclsres=predres(nnetarlgbcls_mean,data,nnetar_pr_mean[,3],lengthtest,h,testindex)
nnetarmeanlgbregres=predres(nnetarlgbreg_mean,data,nnetar_pr_mean[,4],lengthtest,h,testindex)

In [ ]:
end_time = Sys.time()
time_matrix[2,]=end_time-start_time

In [ ]:
end_time-start_time

### The results of the two methods of determining labels are averaged respectively

In [ ]:
nnetarallmin=(nnetarxgbcls_min+nnetarlgbcls_min+nnetarxgbreg_min+nnetarlgbreg_min)/4
nnetarallmean=(nnetarxgbcls_mean+nnetarlgbcls_mean+nnetarxgbreg_mean+nnetarlgbreg_mean)/4

In [ ]:
nnetarallminres=predres_mul(nnetarallmin,data,nnetar_pr_min,lengthtest,h,testindex)
nnetarallmeanres=predres_mul(nnetarallmean,data,nnetar_pr_mean,lengthtest,h,testindex)

### Under the above classification, the improved forecast results of the classification and regression models are averaged respectively

In [ ]:
nnetarclsmin=(nnetarxgbcls_min+nnetarlgbcls_min)/2
nnetarregmin=(nnetarxgbreg_min+nnetarlgbreg_min)/2
nnetarclsmean=(nnetarxgbcls_mean+nnetarlgbcls_mean)/2
nnetarregmean=(nnetarxgbreg_mean+nnetarlgbreg_mean)/2

In [ ]:
nnetarclsminres=predres_mul(nnetarclsmin,data,nnetar_pr_min[,c(1,3)],lengthtest,h,testindex)
nnetarregminres=predres_mul(nnetarregmin,data,nnetar_pr_min[,c(2,4)],lengthtest,h,testindex)
nnetarclsmeanres=predres_mul(nnetarclsmean,data,nnetar_pr_mean[,c(1,3)],lengthtest,h,testindex)
nnetarregmeanres=predres_mul(nnetarregmean,data,nnetar_pr_mean[,c(2,4)],lengthtest,h,testindex)

### Random slice

In [ ]:
nnetarpred_change<- aperm(nnetar_predh[1:datalength,1:m,1:n,1:h], c(1, 3, 2, 4))  
# 使用array函数将其转换为新的形状
nnetarpred_change <- array(nnetarpred_change, dim = c(datalength, m*n, h))

In [ ]:
data$x

In [ ]:
model <- nnetar(data[[1]]$x)
fore_l <- forecast(model, h = h)
nnetarrandomres <- matrix(0, datalength, 6)
for (i in 1:datalength) {
    random <- sample(m * n, n)
    minloc <- min(random)
    # 使用cut函数将数据分割为区间
    loc <- cut(minloc, breaks = seq(0, m * n, n), labels = FALSE)
    loc <- as.numeric(loc)
    nnetarrandompred <- apply(nnetarpred_change[i, random, ], 2, mean)
    start <- length5[i, loc]
    y_all <- data[[i]]$x
    y <- y_all[start:length(y_all)]
    fore_l$x <- ts(y, frequency = freq, end = end(y_all))
    real <- data[[i]]$xx
    fore_l$mean <- ts(nnetarrandompred, start = start(real), frequency = freq)
    res <- accuracy(fore_l, real)
    nnetarrandomres[i, ] <- res[2, 1:6]
}

In [ ]:
mean1=function(x)
    {
# 使用is.finite函数检查哪些值是有限的
finite_values <- x[is.finite(x)]
# 计算有限值的平均值
d=mean(finite_values, na.rm = TRUE)
return(d)
}

In [ ]:
print(apply(nnetar_pred_res[testindex,1,1,],2,mean1))
print(apply(nnetarminxgbclsres,2,mean1))
print(apply(nnetarminxgbregres,2,mean1))
print(apply(nnetarminlgbclsres,2,mean1))
print(apply(nnetarminlgbregres,2,mean1))

In [ ]:
print(apply(nnetarmeanxgbclsres,2,mean1))
print(apply(nnetarmeanxgbregres,2,mean1))
print(apply(nnetarmeanlgbclsres,2,mean1))
print(apply(nnetarmeanlgbregres,2,mean1))

## The actual results used in the paper

In [ ]:
print(apply(nnetarallminres,2,mean1))
print(apply(nnetarallmeanres,2,mean1))

In [ ]:
print(apply(nnetarclsminres,2,mean1))
print(apply(nnetarregminres,2,mean1))
print(apply(nnetarclsmeanres,2,mean1))
print(apply(nnetarregmeanres,2,mean1))

In [ ]:
print(apply(nnetarrandomres[testindex,],2,mean))

In [ ]:
nnetarall <- matrix(0, 16, 6)
nnetarall[1, ] <- apply(nnetar_pred_res[testindex, 1, 1, ], 2, mean1)[1:6]
nnetarall[2, ] <- apply(nnetarminxgbclsres, 2, mean1)
nnetarall[3, ] <- apply(nnetarminxgbregres, 2, mean1)
nnetarall[4, ] <- apply(nnetarminlgbclsres, 2, mean1)
nnetarall[5, ] <- apply(nnetarminlgbregres, 2, mean1)
nnetarall[6, ] <- apply(nnetarmeanxgbclsres, 2, mean1)
nnetarall[7, ] <- apply(nnetarmeanxgbregres, 2, mean1)
nnetarall[8, ] <- apply(nnetarmeanlgbclsres, 2, mean1)
nnetarall[9, ] <- apply(nnetarmeanlgbregres, 2, mean1)
nnetarall[10, ] <- apply(nnetarallminres, 2, mean1)
nnetarall[11, ] <- apply(nnetarallmeanres, 2, mean1)
nnetarall[12, ] <- apply(nnetarclsminres, 2, mean1)
nnetarall[13, ] <- apply(nnetarregminres, 2, mean1)
nnetarall[14, ] <- apply(nnetarclsmeanres, 2, mean1)
nnetarall[15, ] <- apply(nnetarregmeanres, 2, mean1)
nnetarall[16, ] <- apply(nnetarrandomres[testindex, ], 2, mean)

In [49]:
nnetarall

-0.36244547,891.2155,767.9649,-4.456185,15.89326,1.573725
2.55337415,905.2554,773.3460,-3.478981,16.16715,1.346119
-0.05501605,919.9659,791.0185,-2.557098,18.23092,1.352393
13.96135554,904.8248,779.9281,-1.836223,17.75112,1.471512
8.37781124,918.4651,790.6502,-2.365026,18.13537,1.327687
-4.75237402,897.3144,774.1953,-4.561358,16.13061,1.308452
-7.54360101,877.1886,756.7550,-4.659702,16.01769,1.353356
1.42792337,895.8404,772.0605,-4.083381,16.25219,1.464318
-4.82013040,893.8154,773.7723,-4.631482,16.23030,1.348065
6.20938122,855.5866,736.3128,-2.559332,16.50140,1.349840
-3.92204552,845.4409,730.8028,-4.483981,15.38224,1.373036


In [50]:
write.csv(nnetarall,'nnetar_Quarterly_final_res.csv')